# Load packages

In [1]:
import pandas as pd
import pyaging as pya
from itertools import zip_longest

# Load data

In [ ]:
# Path to the directory with clock files (clocks_meta_0.1.30_new.xlsx and clock models)
# Clock models will be downloaded automatically
path_clocks = "E:/YandexDisk/Work/pydnameth/draft/13_fmba_cvd_dnam/ClinEpigen/revision/repo_check/pyaging"
path_save = "path_to_save_results"
df_clocks = pd.read_excel(f"{path_clocks}/pyaging_0.1.30.xlsx", index_col=0)
clocks = df_clocks.index.to_list()

array_type = 'EPICv2' # 'EPICv1' or 'EPICv2' or '450K'

# Load data ('age', 'sex', and betas)
df = pd.read_csv("path_to_file.csv", index_col=0)

# Numerical 'female' column is needed for the different versions of grim clocks and 'dnamfitage'
df['female'] = (df['sex'] == 'F').astype(int)

if array_type == 'EPICv2':
    df = pya.pp.epicv2_probe_aggregation(df, verbose=True)

# Calculate clocks

In [ ]:
# First, we need to calculate grimage separately, since it is required for some other clocks
adata = pya.pp.df_to_adata(df, metadata_cols=['sex'], imputer_strategy='knn', verbose=True)
pya.pred.predict_age(adata=adata, dir=path_clocks, clock_names=['grimage'], verbose=True)
df.loc[df.index, 'grimage'] = adata.obs.loc[df.index, 'grimage']

# Then we can calculate all other clocks
adata = pya.pp.df_to_adata(df, metadata_cols=['sex'], imputer_strategy='knn', verbose=True)
pya.pred.predict_age(adata=adata, dir=path_clocks, clock_names=clocks, verbose=True)

# Save results
results = pd.merge(df[['age', 'sex']], adata.obs[clocks], left_index=True, right_index=True)
results.to_excel(f"{path_save}/clocks.xlsx")

# Save missing features for each clock (number and percentage of missing features, and names of missing features)
missing_features = {}
logger = pya.logger.Logger('test_logger')
device = 'cpu'
for clock in clocks:
    clock_features = list(pya.pred.load_clock(clock, device, path_clocks, logger, indent_level=1).features)
    clock_missed_features = adata.uns[f'{clock}_missing_features']
    clock_percent_na = adata.uns[f'{clock}_percent_na']
    missing_features[clock] = [f"{len(clock_missed_features)} / {len(clock_features)} ({clock_percent_na:0.2f}%)"] + clock_missed_features
pd.DataFrame(zip_longest(*missing_features.values()), columns=missing_features.keys()).to_excel(f"{path_save}/clocks_missing_features.xlsx", index=False)